# ModelSentry Extended V2 Holdout Validation
Runs the frozen v2.4 policy on untouched seeds 314, 2718, and 1618. Select a T4 GPU runtime before starting. Development seeds must not be substituted.

In [ ]:
import platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing')
print('GPU:', torch.cuda.get_device_name(0))

## Load the frozen source from GitHub


In [ ]:
from google.colab import files
from pathlib import Path
import shutil, subprocess
project_dir = Path('/content/ModelSentry')
if project_dir.exists():
    shutil.rmtree(project_dir)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/AT-14/ModelSentry.git', str(project_dir)], check=True)
print('Project directory:', project_dir)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(project_dir / 'requirements-colab.txt')])
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=project_dir, check=True)
print('Preserved CUDA PyTorch:', torch.__version__)

## Run the frozen holdout protocol
This is substantially larger than Baseline V1: four detector modes, four attacks, and 30 benign sessions per mode for each seed.

In [ ]:
output_dir = Path('/content/validation_extended_v2_holdout')
if output_dir.exists():
    shutil.rmtree(output_dir)
command = [sys.executable, 'run_validation_v2.py', '--seeds', '314', '2718', '1618', '--epochs', '8', '--modes', 'rate_only', 'model_aware', 'full', 'enhanced', '--output', str(output_dir)]
subprocess.run(command, cwd=project_dir, check=True)

In [ ]:
import json, pandas as pd
display(pd.read_csv(output_dir / 'per_mode_metrics_v2.csv'))
display(pd.read_csv(output_dir / 'latency_v2.csv'))
summary = json.loads((output_dir / 'validation_summary_v2.json').read_text())
print(json.dumps(summary['mode_overview'], indent=2))

In [ ]:
archive_path = shutil.make_archive('/content/modelsentry_v2_holdout', 'zip', output_dir)
print('Created:', archive_path)
files.download(archive_path)